In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
train = pd.read_csv('train.csv').drop('id', axis=1)
test = pd.read_csv('test.csv').drop('id', axis=1)
sample_submission = pd.read_csv('sample_submission.csv')

In [15]:
for df in [train, test]:

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    # df['MCSquared'] = df['MonthlyCharges'] ** 2
    # df['TCSquared'] = df['TotalCharges'] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    # df['MCSqrt'] = np.sqrt(df['MonthlyCharges'])
    # df['TCSqrt'] = np.sqrt(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    # Loyalty
    df["TenureLog"] = np.log1p(df["tenure"])
    # df["TenureSquared"] = df["tenure"]**2
    # df['TenureSqrt'] = np.sqrt(df['tenure'])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    # Spending
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

    # Contract
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    # Payment
    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # Service Engagement
    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    # Internet
    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # Early Risk
    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    # Value
    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

train['MCGT75%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)
test['MCGT75%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)

train['TCGT75%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)
test['TCGT75%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)

/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_1226/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_1226/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which 

In [16]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [17]:
for feature in X.select_dtypes(include='object'):
    print(feature, X[feature].unique())

gender ['Male' 'Female']
Partner ['Yes' 'No']
Dependents ['Yes' 'No']
PhoneService ['Yes' 'No']
MultipleLines ['No' 'Yes' 'No phone service']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['Yes' 'No' 'No internet service']
OnlineBackup ['No' 'Yes' 'No internet service']
DeviceProtection ['Yes' 'No' 'No internet service']
TechSupport ['Yes' 'No' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['One year' 'Two year' 'Month-to-month']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Mailed check' 'Credit card (automatic)' 'Electronic check'
 'Bank transfer (automatic)']


In [18]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['Contract', 'PaymentMethod']

ord_mapping = {
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

# --- Применяем трансформацию ---
X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [19]:
X_train_encoded.shape

(594194, 40)

In [20]:
pipeline.get_feature_names_out()

array(['onehot__Contract_One year', 'onehot__Contract_Two year',
       'onehot__PaymentMethod_Credit card (automatic)',
       'onehot__PaymentMethod_Electronic check',
       'onehot__PaymentMethod_Mailed check', 'ordinal__MultipleLines',
       'ordinal__InternetService', 'ordinal__OnlineSecurity',
       'ordinal__OnlineBackup', 'ordinal__DeviceProtection',
       'ordinal__TechSupport', 'ordinal__StreamingTV',
       'ordinal__StreamingMovies', 'ordinal__Partner',
       'ordinal__Dependents', 'ordinal__PhoneService',
       'ordinal__PaperlessBilling', 'ordinal__gender',
       'remainder__SeniorCitizen', 'remainder__tenure',
       'remainder__MonthlyCharges', 'remainder__TotalCharges',
       'remainder__MCLog', 'remainder__TCLog', 'remainder__MCSqueezed',
       'remainder__TCSqueezed', 'remainder__TenureLog',
       'remainder__TenureSqueezed', 'remainder__AvgMonthlySpend',
       'remainder__SpendDeviation', 'remainder__IsMonthToMonth',
       'remainder__IsElectronicCheck',

In [21]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        eval_metric="auc"
    )

    model.fit(X_train, y_train)
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [22]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9161805979802863


In [23]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)